# 04: Planted-rule population and exact-risk oracle

Design decisions
- Population: 20,000 rows drawn once, without replacement and with seed 42, from the 21 predictors of the BRFSS 2015 heart table (notebook 01). Every later training, calibration and test set of the validity study is drawn with replacement from this fixed population, so they are independent draws from a known distribution.
- Labels are generated from a known probability p(x). Rules P1 to P3 use p(x) = eps + (1 - 2 eps) 1[rule(x)] with label noise eps in {0, 0.05, 0.10, 0.20}; rule prevalence is the share of the population inside the rule. Rule P4 is a smooth logistic risk with no exact rule; its intercept is set so that the population mean of p is 0.10.
- Exact-risk oracle: the true risk of a box (plan Eq. 13) and the true selective risk of a release policy (plan Eq. 14) are averages over the 20,000 population rows, so they carry no estimation error.
- The population row indices are committed (`results/04_population_indices.csv`); the population table itself stays on Drive.
- Checks: a box equal to the P1 rule region has risk exactly eps; the whole space has risk equal to the prevalence; the oracle agrees with simulated labels on large draws; the selective-risk oracle reproduces the 10/105 overlap example.

In [ ]:
# === SESSION BOOTSTRAP (run first) ===
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'statsmodels'], check=True)
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
DRIVE_ROOT = '/content/drive/MyDrive/SCAX_Research'
REPO_DIR = f'{DRIVE_ROOT}/scax-research'
for name in ('.gitconfig', '.git-credentials'):
    if os.path.exists(f'{DRIVE_ROOT}/{name}'):
        shutil.copy(f'{DRIVE_ROOT}/{name}', os.path.expanduser(f'~/{name}'))
os.chdir(REPO_DIR)
os.environ['SCAX_DRIVE_ROOT'] = DRIVE_ROOT
sys.path.insert(0, f'{REPO_DIR}/src')
pull = subprocess.run(['git', 'pull', 'origin', 'main'], capture_output=True, text=True)
print(pull.stdout.strip() or pull.stderr.strip())

In [ ]:
import json
import numpy as np
import pandas as pd
from scax import paths
from scax.utils import seed_everything, sha256_array, sha256_file, write_json
from scax.data import public, planted

SEED = 42
N_POP = 20000
seed_everything(SEED)
paths.ensure_dirs()
X_all, _ = public.load_brfss_heart(paths.RAW)
SOURCE = paths.RAW / 'brfss2015_heart' / public.SPECS['brfss2015_heart'].file
IDX = planted.make_population(len(X_all), N_POP, SEED)
XP = X_all.iloc[IDX].reset_index(drop=True)
pd.DataFrame({'source_row': IDX}).to_csv(f'{paths.RESULTS}/04_population_indices.csv', index=False)
XP.to_parquet(paths.DERIVED / 'planted_population.parquet', index=False)
print('population:', XP.shape, '| distinct covariate rows:', int(XP.drop_duplicates().shape[0]))

## Label probabilities for every rule and noise level

In [ ]:
B0_P4 = planted.p4_intercept(XP)
P = {}
rows = []
for rule in ('P1', 'P2', 'P3'):
    inside = planted.rule_mask(rule, XP)
    for eps in planted.NOISE_LEVELS:
        p = planted.probability(rule, XP, eps)
        P[f'{rule}_eps{eps:.2f}'] = p
        rows.append({'rule': rule, 'eps': eps, 'rule_share': float(inside.mean()), 'prevalence': float(p.mean()),
                     'features': ' '.join(sorted(planted.RULE_FEATURES[rule]))})
P['P4'] = planted.probability('P4', XP, p4_b0=B0_P4)
rows.append({'rule': 'P4', 'eps': None, 'rule_share': None, 'prevalence': float(P['P4'].mean()),
             'features': ' '.join(sorted(planted.RULE_FEATURES['P4']))})
PREV = pd.DataFrame(rows)
pd.DataFrame(P).to_parquet(paths.DERIVED / 'planted_probabilities.parquet', index=False)
print('P4 intercept:', round(B0_P4, 4))
print(PREV.to_string(index=False))
assert all(0 <= v.min() and v.max() <= 1 for v in P.values())

## Oracle checks

In [ ]:
CHECKS = {}
p1_box = planted.Box({'HighBP': (1, 2), 'Age': (10, np.inf)})
assert np.array_equal(p1_box.contains(XP), planted.rule_mask('P1', XP))
CHECKS['p1_rule_box_risk_equals_eps'] = {
    f'{eps:.2f}': planted.true_box_risk(p1_box, 1, XP, P[f'P1_eps{eps:.2f}']) for eps in planted.NOISE_LEVELS}
assert all(abs(v - float(k)) < 1e-12 for k, v in CHECKS['p1_rule_box_risk_equals_eps'].items())

whole = planted.Box({})
CHECKS['whole_space_risk_equals_prevalence'] = all(
    abs(planted.true_box_risk(whole, 0, XP, p) - p.mean()) < 1e-12 for p in P.values())
assert CHECKS['whole_space_risk_equals_prevalence']

# simulated labels on large with-replacement draws agree with the oracle within four standard errors
rng = np.random.default_rng(SEED)
boxes = [planted.Box({'Age': (9, np.inf)}), planted.Box({'GenHlth': (4, np.inf), 'DiffWalk': (1, 2)}),
         planted.Box({'BMI': (30, np.inf), 'HighChol': (1, 2)}), planted.Box({'Age': (1, 6), 'HighBP': (0, 1)})]
sim = []
for key in ('P2_eps0.10', 'P3_eps0.05', 'P4'):
    p = P[key]
    draw = planted.draw_with_replacement(N_POP, 400000, rng)
    y = planted.draw_labels(p[draw], rng)
    for b in boxes:
        for y_box in (0, 1):
            m = b.contains(XP.iloc[draw])
            emp = float(np.mean(y[m] != y_box))
            exact = planted.true_box_risk(b, y_box, XP, p)
            se = float(np.sqrt(exact * (1 - exact) / m.sum()))
            sim.append({'labels': key, 'box': str(b.bounds), 'y_box': y_box, 'n_in_box': int(m.sum()),
                        'simulated': emp, 'exact': exact, 'z': (emp - exact) / se if se > 0 else 0.0})
SIM = pd.DataFrame(sim)
print(SIM.round(4).to_string(index=False))
assert (SIM['z'].abs() < 4).all(), 'oracle and simulation disagree'
CHECKS['simulation_max_abs_z'] = float(SIM['z'].abs().max())

p_overlap = np.r_[np.ones(10), np.zeros(95)]
CHECKS['overlap_example_selective_risk'] = planted.true_selective_risk(np.ones(105, bool), np.zeros(105, int), p_overlap)
assert abs(CHECKS['overlap_example_selective_risk'] - 10 / 105) < 1e-12
print(json.dumps(CHECKS, indent=1))

In [ ]:
SIM.to_csv(f'{paths.RESULTS}/04_oracle_simulation.csv', index=False)
PREV.to_csv(f'{paths.RESULTS}/04_rule_prevalence.csv', index=False)
write_json(f'{paths.RESULTS}/04_population_summary.json', {
    'seed': SEED, 'population_size': N_POP, 'source_file_sha256': sha256_file(SOURCE),
    'population_indices_sha256': sha256_array(IDX), 'population_matrix_sha256': sha256_array(XP.to_numpy()),
    'distinct_covariate_rows': int(XP.drop_duplicates().shape[0]), 'p4_intercept': B0_P4,
    'p4_target_prevalence': planted.P4_TARGET_PREVALENCE, 'noise_levels': list(planted.NOISE_LEVELS),
    'rule_prevalence': PREV.to_dict('records'), 'oracle_checks': CHECKS})
print('written: results/04_population_indices.csv, 04_rule_prevalence.csv, 04_oracle_simulation.csv, 04_population_summary.json')

In [ ]:
# === STANDARD COMMIT CELL ===
import os, subprocess
os.chdir(REPO_DIR)
r = subprocess.run(["python", "tools/commit_cell.py",
  "04: planted-rule population (20,000 BRFSS 2015 rows), label probabilities for P1 to P4, exact-risk oracle checks"],
  capture_output=True, text=True)
print(r.stdout); print(r.stderr)